# 4.6 Proteomics: Blood Proteins in Type 2 Diabetes

**Omics series, class 6 of 7.** DNA (4.1–4.4) is the instruction book; RNA (4.5) shows which instructions are being read; **proteins** are what the body actually builds and uses: hormones, enzymes, signals. *All of Us* has measured about **5,400 proteins** in blood from about **10,000 participants**, with a technology called **Olink**.

Our questions:

1. **Which blood proteins are higher or lower in people with type 2 diabetes?**
2. **Are those proteins controlled by genetic variants (pQTLs)?** And does rs7903146 affect any protein?

### By the end of this notebook you'll be able to

1. Explain how Olink measures proteins, and what an **NPX** value is
2. Find the *All of Us* proteomics files, and read only the rows you need from a very large file
3. Apply **QC** flags, and reshape long data into a protein × person matrix
4. Run a **differential abundance** analysis with `limma`, and read the results
5. Look up **pQTLs** (variants linked to protein levels) in the *All of Us* results

| | |
|---|---|
| 📂 **Uses** | the *All of Us* proteomics files (v9) and the Controlled Tier dataset |
| 💾 **Saves** | `proteomics_t2d_results.csv`: one row per protein |

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Questions** | Which **plasma proteins** differ in type 2 diabetes, and are their levels set partly by **genetics**? |
| 🧪 **The analysis** | **Differential abundance**: a **linear model** per protein with limma on NPX, adjusted for age and sex, after QC and checks of the **normalization** and **batch** (plate) effects. Then a **pQTL** lookup |
| 🎯 **Why** | NPX is already normalized and on a log₂ scale, so linear models fit directly; but we **check** the normalization worked, because leftover plate effects can look like biology; ~5,400 proteins, so an adjusted p-value (FDR) |
| ✅ **What we conclude** | Which proteins differ and which **organs** they point to (in our test run: hundreds of proteins, pointing to the kidney, fat tissue, and gut); which have strong pQTLs |
| ⚠️ **What it can't tell us** | Cause vs consequence vs treatment; for sure which organ a protein came from |
| 🔗 **Connections** | RNA in **4.5**; both together in **4.7**. Combining pQTLs with a GWAS (**4.2**) is the idea behind **Mendelian randomization**: does a protein **cause** the disease? |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts | 15 |
| 2. Setup and a tour of the proteomics files | 15 |
| 3. The cohort | 10 |
| 4. Load our people's proteins | 15 |
| 5. QC, reshape, explore | 20 |
| 6. Differential abundance | 25 |
| 7. pQTLs | 15 |
| Wrap-up | 5 |

**Before you start:**

- This notebook works **on its own**. It's an **R** notebook in `notebooks/omics/`, and loads `../aou_setup.R`.
- It uses **limma**, installed by the setup cell (as in 4.5; quick if you've run 4.5).

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 120 minutes |
| ⏱️ **Run time** | about **10 minutes** to run every cell (plus a few minutes the first time, for the optional enrichment packages). Slowest: loading the proteins (3–5 min, 3.5 in our test) and the cohort query (1–2 min) |
| 💰 **Data cost** | **under $1**: one BigQuery query, plus reading the 2.6 GB protein file from the *All of Us* bucket (pennies) |
| 💰 **App cost** | about **$0.27 an hour** while the app is running (the default app: 4 CPUs, 26 GB memory), so about **$0.55** for a 2-hour class. The app's disk costs about **$0.70 a day**, even while the app is stopped |
| 💡 **To spend less** | don't rerun the loading cell (it reads the whole file each time); stop the app during long breaks |
| **Estimated total** | about **$1 to $2** per student for this class |

App prices are the Workbench's estimate for the default app (October 2026); yours are on the **Apps** tab when you create or edit an app. BigQuery and bucket costs are our estimates from test runs. **Stop** the app after each class, and **delete** it when you've finished the series (save anything you need to your bucket first: deleting removes everything on the app's disk).

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data (including any person's protein values), or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### From genes to proteins

| | DNA (4.1–4.4) | RNA (4.5) | Protein (today) |
|---|---|---|---|
| What it is | the instructions | a working copy of one gene | the product |
| Changes with disease? | no (you're born with it) | yes | yes |
| Measured in *All of Us* | genomes, arrays | RNA-seq, whole blood | **Olink**, blood plasma |
| Participants (v9) | about 535,000 | about 9,000 | about 10,000 |

Proteins in the blood come from many organs (liver, fat, muscle, the immune system, the pancreas), so they can reflect what's happening **throughout the body**, not only in blood cells. That's one reason proteins are promising **biomarkers**.

### How Olink measures proteins

Olink uses **pairs of antibodies**. Each antibody sticks to a different part of one protein, and each carries a short **DNA tag**. Only when **both** antibodies bind the same protein molecule are their tags close enough to join into a short DNA sequence, which is then counted by **DNA sequencing**. More protein → more joined tags → more counts.

- The *All of Us* platform is **Olink Explore HT**: about 5,400 proteins (**assays**) from one small blood sample.
- Each assay has an **OlinkID** (e.g. `OID20049`), the protein's **UniProt** ID, and a short name (**Assay**, usually the gene name, e.g. `LEP` for leptin).

### NPX: Olink's unit

Olink reports **NPX** (*Normalized Protein eXpression*):

- It's on a **log₂ scale**, like log-CPM in 4.5: a difference of **1** means **twice** as much protein.
- It's **relative**, not an absolute concentration: you can compare NPX for the **same protein** between people, but not between two different proteins.
- It's already **normalized**, in two steps, before we see it:
  1. **Plate-control normalization** (by Olink's software): each plate includes control samples, and values are adjusted so the controls match across plates (`Normalization` = *Plate control* in the file).
  2. **Batch normalization** (by *All of Us*, hence the folder name `batch_normalized`): samples were run in batches over many months, and the batches were adjusted to agree with each other. This is usually done with **bridging samples** measured in more than one batch; the file's `Adj_factor` column appears to hold the adjustment. Check the *All of Us* proteomics documentation for the exact method.
- Because of that, we **don't** normalize again. **limma doesn't normalize anything**: it fits models to the values you give it. But we still **check** that the normalization worked (Part 5): leftover differences between plates (**batch effects**) can look like biology.

### Limit of detection

Every assay has a **limit of detection (LOD)**: below it, the signal can't be told apart from background noise. Some proteins are near or below their LOD in many people; their NPX values are mostly noise, and differences between groups are hard to trust. Olink reports LODs, but the *All of Us* file we use doesn't seem to include an LOD column, so we can't filter on it here. If a "significant" protein has a very **low average NPX** (`AveExpr`) compared with others, keep LOD in mind.

### Differential abundance

The same idea as differential expression in 4.5: for **each protein**, compare cases with controls, adjusted for age and sex, and correct for testing ~5,400 proteins (**adjusted p-value**, FDR). We use **limma** again, but without `voom()`: NPX values are already on a log scale and aren't counts.

### pQTLs

A **pQTL** (*protein quantitative trait locus*) is a variant linked to a protein's level: the protein version of an eQTL (4.5). **Cis**-pQTLs are near the protein's own gene. *All of Us* has calculated them, for everyone combined (**COMB**) and per ancestry group.

### Glossary

| Term | Means |
|---|---|
| **Proteomics** | measuring many proteins at once |
| **Plasma** | the liquid part of blood, without the cells |
| **Assay** | one protein measurement on the Olink panel |
| **NPX** | Olink's relative, log₂-scale protein level |
| **Long format** | one row per person per protein (vs **wide**: one column per person) |
| **Differential abundance** | a difference in a protein's level between groups |
| **pQTL** | a variant linked to a protein's level |
| **Batch effect** | a technical difference between groups of samples processed together (e.g. on one plate) |
| **LOD** | limit of detection: the lowest level an assay can reliably measure |

## Part 2. Setup and a tour of the proteomics files

The setup cell loads `aou_setup.R` and, if needed, installs **limma** from Bioconductor (5–10 minutes the first time; quick if you ran 4.5).

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

# limma, from Bioconductor (first time only)
user_lib <- Sys.getenv("R_LIBS_USER")
dir.create(user_lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(user_lib, .libPaths()))
for (pkg in c("data.table", "BiocManager")) {
  if (!requireNamespace(pkg, quietly = TRUE)) install.packages(pkg, lib = user_lib, repos = "https://cloud.r-project.org")
}
if (!requireNamespace("limma", quietly = TRUE)) BiocManager::install("limma", lib = user_lib, update = FALSE, ask = FALSE)
suppressPackageStartupMessages(library(limma))

PROT_DIR <- file.path(aou_data_dir("v9"), "multiomics", "proteomics")
list.files(PROT_DIR)
list.files(file.path(PROT_DIR, "npx", "batch_normalized"))

**🔍 Read the output:**

| File or folder | What's in it |
|---|---|
| `manifest.tsv` | one row per **sample**: its participant (`research_id`), plate, and file locations |
| `npx/batch_normalized/` | **all** samples in one file, batch-normalized across plates. `…_reps_removed` has **one sample per person** (repeat measurements removed): **what we use** |
| `npx/tsv/`, `npx/parquet/` | the original files, one per **batch** of plates (`LCSET-…`), before batch normalization |
| `olink_report/` | Olink's PDF quality report for each batch |
| `pqtl/` | precomputed **pQTL** results (Part 7) |

### The manifest

Column names and counts only: the rows are participant IDs.

In [ ]:
manifest <- read_tsv(file.path(PROT_DIR, "manifest.tsv"), col_types = cols(.default = "c"))

names(manifest)
nrow(manifest)                     # samples
n_distinct(manifest$research_id)   # participants

**🔍 Read the output:** there are a few more **samples** than **participants**: some people were measured twice (**replicates**, to check the measurements are repeatable). The `_reps_removed` file keeps one sample per person.

### The protein file

`Olink_10k_aou_v9_reps_removed.tsv.gz` is in **long format**: one row per person **per protein**. That's about 10,000 people × 5,400 proteins: **over 50 million rows**, 2.6 GB compressed. Far too big to load whole.

We read just the **first line** (the column names) with a command-line pipe: `gzip -dc` uncompresses, `head -1` stops after one line.

In [ ]:
NPX_FILE <- file.path(PROT_DIR, "npx", "batch_normalized", "Olink_10k_aou_v9_reps_removed.tsv.gz")

npx_columns <- str_split(system(str_glue("gzip -dc {shQuote(NPX_FILE)} | head -1"), intern = TRUE), "\t")[[1]]
npx_columns

**🔍 Read the output:** 36 columns. The ones we need:

| Column | Means |
|---|---|
| `ResearchID` | the participant (same as `person_id`) |
| `SampleID` | the sample |
| `PlateID` | the plate the sample was measured on (used to check for batch effects) |
| `OlinkID`, `UniProt`, `Assay` | the protein: Olink's ID, the UniProt ID, and its short name (usually the gene) |
| `NPX` | the protein level (log₂ scale) |
| `SampleQC` | `PASS` or `FAIL`: did this sample's measurement in this block of the panel pass Olink's checks? |
| `AssayQC` | `PASS` or not: did the protein's assay pass on this plate? |

Others describe the plate (`PlateID`, `WellID`), the panel's **blocks** (Explore HT is run in blocks of assays), alternative NPX versions (`ExtNPX`, `PCNormalizedNPX`), and software versions.

## Part 3. The cohort

The same type 2 diabetes cohort as in 4.1–4.5, kept to people with proteomics, aged 18 to 100, sex at birth *Female* or *Male*. Then 300 cases and 300 controls at random.

💰 The query costs cents to under $1.

In [ ]:
cohort <- run_query(str_glue(read_file("../sql/t2d_cohort.sql"), AGE_DATE = "2025-01-01")) %>%
  mutate(person_id = as.character(person_id))

# Everyone with proteomics, by cohort group
cohort %>% filter(person_id %in% manifest$research_id) %>% count(cohort) %>% hide_small_counts()

prot_cohort <- cohort %>%
  filter(person_id %in% manifest$research_id,
         cohort %in% c("case", "control"),
         between(age, 18, 100),
         sex_at_birth %in% c("Female", "Male")) %>%
  mutate(group = factor(cohort, levels = c("control", "case")))

N_PER_GROUP <- 300
set.seed(2026)
samples <- prot_cohort %>%
  arrange(person_id) %>%
  group_by(group) %>%
  slice_sample(n = N_PER_GROUP) %>%
  ungroup()

samples %>%
  group_by(group) %>%
  summarise(n = n(), mean_age = round(mean(age), 1), pct_female = round(100 * mean(sex_at_birth == "Female"), 1))

**🔍 Read the output:** how many people with proteomics are cases, controls, or excluded; then our 300 + 300. As in 4.5, cases are **older** (64 vs 52 in our test run) and less often female, so age and sex go in the model.

## Part 4. Load our people's proteins

We use **awk**, a small command-line tool for text tables, to keep only **our 600 people's rows** and **the columns we need**, before R sees anything:

| Part of the command | What it does |
|---|---|
| `gzip -dc file` | uncompresses the file as it's read |
| `awk … 'NR==FNR {keep[$1]; next}'` | first reads `ids.txt` (our 600 IDs) into a list called `keep` |
| `FNR==1 \|\| ($col in keep)` | then keeps the header line, and every line whose `ResearchID` is in `keep` |
| `{print $a, $b, …}` | prints only the columns we need |

`match()` finds each column's **position** in the header, because awk refers to columns by number (`$27`), not name.

`fread(cmd = …)` (from **data.table**, as in 4.5) runs the command and reads the result.

⏱️ **3 to 5 minutes** (3.5 in our test run): the whole 2.6 GB file is uncompressed and scanned once. A good moment for questions.

In [ ]:
ids_file <- file.path(tempdir(), "ids.txt")
write_lines(samples$person_id, ids_file)

wanted   <- c("ResearchID", "SampleID", "PlateID", "OlinkID", "UniProt", "Assay", "NPX", "SampleQC", "AssayQC")
col_nums <- match(wanted, npx_columns)
stopifnot(!anyNA(col_nums))   # stops if a column name has changed

awk_command <- str_glue(
  "gzip -dc {shQuote(NPX_FILE)} | ",
  "awk -F'\\t' -v OFS='\\t' -v id={col_nums[1]} ",
  "'NR==FNR {{keep[$1]; next}} FNR==1 || ($id in keep) {{print {str_c('$', col_nums, collapse = ', ')}}}' ",
  "{shQuote(ids_file)} -"
)

start <- Sys.time()
npx_long <- data.table::fread(cmd = awk_command, sep = "\t", header = TRUE,
                              colClasses = c(ResearchID = "character", NPX = "numeric")) %>%
  as_tibble()
Sys.time() - start

dim(npx_long)
n_distinct(npx_long$ResearchID)   # people found
n_distinct(npx_long$OlinkID)      # proteins

**🔍 Read the output:** about 600 × 5,400 = 3 million rows, our 600 people (if a few are missing, their samples weren't in this file), and about 5,400 proteins.

The doubled braces `{{ }}` in the command are how `str_glue()` writes a literal `{`: single braces are for inserting R values.

## Part 5. QC, reshape, explore

### QC flags

Each measurement is labelled `PASS`, `WARN`, or `FAIL`. Olink's usual advice: **drop `FAIL`**, keep `WARN` (usable, with a minor issue). We count them, then set failed measurements to missing (`NA`).

In [ ]:
npx_long %>% count(SampleQC, AssayQC)   # rows (person x protein), not people

# People with at least one failed block (don't share if 1 to 20)
npx_long %>% filter(SampleQC == "FAIL") %>% summarise(people = n_distinct(ResearchID)) %>%
  rename(n = people) %>% hide_small_counts()

npx_long <- npx_long %>%
  mutate(NPX = if_else(SampleQC == "FAIL" | AssayQC == "FAIL", NA_real_, NPX))

**🔍 Read the output:** most rows **pass** (in our test run, over 99%; under 1% failed, and a few were `WARN`). A sample can fail in one **block** of the panel and pass in the others, so we drop failed measurements, not whole people.

### Long to wide

limma wants a **matrix**: one row per **protein**, one column per **person** (like the count matrix in 4.5). `pivot_wider()` turns the long table into that shape (see **2.2 reshape**, coming soon in the core series). We name rows by `OlinkID`, which is unique; `Assay` names can repeat.

In [ ]:
protein_info <- npx_long %>% distinct(OlinkID, UniProt, Assay)

npx_wide <- npx_long %>%
  select(OlinkID, ResearchID, NPX) %>%
  pivot_wider(names_from = ResearchID, values_from = NPX, values_fn = mean)

npx <- as.matrix(npx_wide[, -1])
rownames(npx) <- npx_wide$OlinkID

# Put the sample table in the same order as the columns
samples <- samples %>% filter(person_id %in% colnames(npx))
npx     <- npx[, samples$person_id]
stopifnot(identical(colnames(npx), samples$person_id))

dim(npx)   # proteins x people

# Share of missing values per protein
missing_share <- rowMeans(is.na(npx))
summary(missing_share)

**🔍 Read the output:** about 5,400 proteins × 600 people. Most proteins have few or no missing values (from failed QC); a few are missing for **everyone** (`Max.` = 1): they failed on every plate.

We keep proteins measured in at least **90%** of our people. limma can handle the remaining gaps.

In [ ]:
npx <- npx[missing_share <= 0.10, ]
dim(npx)

# NPX for one person: roughly centred on 0, on a log2 scale
hist(npx[, 1], breaks = 50, main = "NPX values, one person", xlab = "NPX (log2 scale)")

### PCA

As in 4.5: the big patterns. PCA can't handle missing values, so **for the PCA only** we fill each gap with that protein's average (a simple **imputation**). The differential abundance test in Part 6 uses the real values, gaps and all.

In [ ]:
filled <- npx
gaps   <- which(is.na(filled), arr.ind = TRUE)                    # row and column of each gap
filled[gaps] <- rowMeans(npx, na.rm = TRUE)[gaps[, "row"]]       # the protein's average

pca <- prcomp(t(filled), scale. = TRUE)
pct <- round(100 * pca$sdev^2 / sum(pca$sdev^2), 1)

pca_df <- tibble(PC1 = pca$x[, 1], PC2 = pca$x[, 2], group = samples$group,
                 sex_at_birth = samples$sex_at_birth, age = samples$age)

ggplot(pca_df, aes(PC1, PC2, colour = group)) +
  geom_point(alpha = 0.6) +
  labs(x = str_glue("PC1 ({pct[1]}%)"), y = str_glue("PC2 ({pct[2]}%)"), colour = NULL,
       title = "PCA of blood proteins")

**🔍 Read the output:** do cases and controls separate? As with RNA, usually only a little: diabetes changes some proteins, but it isn't the biggest source of variation. `scale. = TRUE` puts every protein on the same scale first, so no single protein dominates.

In our test run, **two people** sat far away from everyone else. **Outliers** like these can be a sample problem (e.g. a badly handled tube) or a person with an unusual condition. Before removing anyone, check their QC flags and whether removing them changes your results; never remove people just because they make the results look worse.

### ✍️ Your turn

Colour the PCA by **age** instead (`colour = age`). Does PC1 or PC2 follow age?

<details>
<summary><b>Show answer</b></summary>

```r
ggplot(pca_df, aes(PC1, PC2, colour = age)) +
  geom_point(alpha = 0.6)
```

Many blood proteins change with **age** (e.g. *GDF15*), so age often lines up with a main PC. That's one more reason to adjust for it.

</details>

In [ ]:
# Your code here

### Check the normalization and batch effects

The data was normalized before release, but a careful analyst **checks**. Three questions:

1. **Do the plates line up?** After normalization, each person's **median** NPX (across all proteins) should be about the same on every plate. A plate whose people sit much higher or lower has a leftover **batch effect**.
2. **Does plate drive the main patterns?** For each of the first 5 PCs, how much of it does **plate** explain, compared with diabetes, age, and sex? We use the **adjusted R²** of a regression of each PC on each variable: the share of the PC explained, corrected for the number of categories (there are many plates, and more categories always explain a little by chance).
3. **Are there outlier samples?** A rule instead of eyeballing: flag anyone more than **5 standard deviations** from the average on PC1 or PC2.

In [ ]:
# Each person's plate (one plate per sample)
samples <- samples %>%
  left_join(npx_long %>% distinct(person_id = ResearchID, PlateID) %>% distinct(person_id, .keep_all = TRUE),
            by = "person_id")
n_distinct(samples$PlateID)   # plates our people were measured on

# 1. Each person's median NPX, by plate
tibble(median_npx = apply(npx, 2, median, na.rm = TRUE), PlateID = samples$PlateID) %>%
  ggplot(aes(reorder(PlateID, median_npx), median_npx)) +
  geom_boxplot(outlier.shape = NA) +
  theme_bw() +
  theme(axis.text.x = element_blank(), axis.ticks.x = element_blank()) +
  labs(x = "Plate (one box per plate)", y = "Median NPX per person", title = "Do the plates line up after normalization?")

# 2. How much of each PC does each variable explain? (adjusted R-squared)
pc_data <- as_tibble(pca$x[, 1:5]) %>% bind_cols(samples %>% select(PlateID, group, age, sex_at_birth))
explained <- function(pc, variable) summary(lm(reformulate(variable, pc), data = pc_data))$adj.r.squared
tibble(PC = paste0("PC", 1:5)) %>%
  mutate(plate    = map_dbl(PC, explained, "PlateID"),
         diabetes = map_dbl(PC, explained, "group"),
         age      = map_dbl(PC, explained, "age"),
         sex      = map_dbl(PC, explained, "sex_at_birth")) %>%
  mutate(across(-PC, ~ round(.x, 3)))

# 3. Outlier samples: more than 5 SD from the average on PC1 or PC2
samples <- samples %>%
  mutate(outlier = abs(as.numeric(scale(pca$x[, 1]))) > 5 | abs(as.numeric(scale(pca$x[, 2]))) > 5)
sum(samples$outlier)   # if 1 to 20, don't share the number

**🔍 Read the output:**

1. **The boxplot:** the boxes should sit at about the same height, with no plate far above or below the rest. A few uneven boxes are normal when a plate holds only a handful of our 600 people. **Careful:** `reorder()` sorts the plates by their median, so the boxes **always** rise from left to right: look at **how much** they rise. In our test run (120 plates), the medians spread over about half an NPX unit, with no plate clearly apart from the rest.
2. **The table:** plate should explain **little** of each PC (adjusted R² near 0). If plate explains more than diabetes or age, batch effects remain, and plate belongs in the model (we test that in Part 6). In our test run, plate's adjusted R² was **at or below 0** for all five PCs (negative values happen when a variable explains less than its number of categories would by chance). Age explained the most, and only about 8% of PC2.
3. **Outliers:** in our test run, a few people (fewer than 20) were flagged. We **don't** delete them silently: in Part 6 we check whether removing them changes the results (a **sensitivity analysis**). If it doesn't, keep them; if it does, investigate before deciding, and report what you did.

## Part 6. Differential abundance

For every protein: **NPX ~ group + age + sex at birth**, testing `groupcase`. NPX is already on a log₂ scale, so we use `lmFit()` directly (no `voom()`, which is for counts).

In [ ]:
design <- model.matrix(~ group + age + sex_at_birth, data = samples)

fit     <- eBayes(lmFit(npx, design))
results <- topTable(fit, coef = "groupcase", number = Inf) %>%
  rownames_to_column("OlinkID") %>%
  left_join(protein_info, by = "OlinkID") %>%
  as_tibble()

results %>%
  select(Assay, OlinkID, logFC, AveExpr, P.Value, adj.P.Val) %>%
  head(20)

results %>%
  filter(adj.P.Val < 0.05) %>%
  count(direction = if_else(logFC > 0, "higher with diabetes", "lower with diabetes"))

**🔍 Read the output:**

| Column | Means |
|---|---|
| `Assay` | the protein's short name |
| `logFC` | the difference in NPX, **case − control**: 1 = twice as much in people with diabetes |
| `AveExpr` | the protein's average NPX |
| `adj.P.Val` | the **adjusted p-value** (FDR) across all proteins. **Use this one.** |

Proteins tend to show **more** differences than blood RNA: they come from organs all over the body. Look for proteins linked to **insulin resistance**, **fat tissue**, **the liver**, **the kidneys**, and **inflammation**.

### Volcano plot

In [ ]:
ggplot(results, aes(logFC, -log10(P.Value), colour = adj.P.Val < 0.05)) +
  geom_point(size = 0.8, alpha = 0.6) +
  geom_text(data = head(results, 10), aes(label = Assay), vjust = -0.6, size = 3,
            check_overlap = TRUE, show.legend = FALSE) +
  scale_colour_manual(values = c(`FALSE` = "grey60", `TRUE` = "#08519c"), name = "Adjusted p < 0.05") +
  geom_vline(xintercept = 0, linetype = "dashed") +
  theme_bw() +
  labs(x = "logFC (NPX, type 2 diabetes vs control)", y = "-log10(p-value)",
       title = "Blood proteins in type 2 diabetes")

**🔍 Read the output:** proteins high and far to the right are clearly **higher** in diabetes; far to the left, **lower**. The 10 most significant are labelled.

### What we found (test run)

About **290** proteins differed (adjusted p < 0.05), most of them **higher** in diabetes, compared with a few dozen genes in blood RNA (4.5). Look at which **organs** they point to:

| Organ | Proteins (test run) | A possible story |
|---|---|---|
| **Kidney** | *HAVCR1* (KIM-1, a marker of kidney injury) and *PDZK1* higher; *UMOD* (uromodulin, made only by the kidney) lower | early **diabetic kidney disease** |
| **Fat tissue** | *FABP4* higher; *LPL* (lipoprotein lipase) lower | **insulin resistance** in fat tissue |
| **Gut** | *LGALS4*, *MUC13*, *PRAP1* higher | possibly **medicines** such as metformin, which acts in the gut |
| **Nerves** | *NFASC* (the top protein) | to explore: diabetes often damages nerves |

**💬 Discuss:** which of these could be **causes** of diabetes, which **consequences** (e.g. kidney damage), and which **treatment** effects? How could you tell? (Ideas: kidney function (**eGFR**, calculated from creatinine in the `measurement` table) as a covariate for the kidney proteins; medication records in the `drug_exposure` table; when the blood sample was taken compared with the diagnosis; pQTLs and Mendelian randomization, Part 7.)

### The top protein

In [ ]:
top <- results$OlinkID[1]

tibble(NPX = npx[top, ], group = samples$group) %>%
  filter(!is.na(NPX)) %>%
  ggplot(aes(group, NPX)) +
  geom_boxplot() +
  theme_bw() +
  labs(title = results$Assay[1], y = "NPX (log2 scale)", x = NULL)

In [ ]:
save_data(results, "proteomics_t2d_results.csv")

### Sensitivity checks: plate, and outliers

A **sensitivity analysis** reruns the analysis with one choice changed, to see whether the conclusions hold. Two checks:

- **Plate in the model:** add `PlateID` as a covariate, which removes any average difference between plates.
- **Without the outliers** flagged in Part 5.

For each, we count the significant proteins and compare the logFCs with the main results (a correlation near 1 means "the same answer").

In [ ]:
compare <- function(fit_new, label) {
  new <- topTable(fit_new, coef = "groupcase", number = Inf) %>% rownames_to_column("OlinkID")
  both <- inner_join(results %>% select(OlinkID, logFC), new %>% select(OlinkID, logFC_new = logFC), by = "OlinkID")
  tibble(analysis = label, significant = sum(new$adj.P.Val < 0.05),
         logFC_correlation = round(cor(both$logFC, both$logFC_new), 3))
}

# Plate as a covariate
design_plate <- model.matrix(~ group + age + sex_at_birth + PlateID, data = samples)
fit_plate    <- eBayes(lmFit(npx, design_plate))

# Without the outliers
keep        <- !samples$outlier
design_keep <- model.matrix(~ group + age + sex_at_birth, data = samples[keep, ])
fit_keep    <- eBayes(lmFit(npx[, keep], design_keep))

bind_rows(
  tibble(analysis = "main", significant = sum(results$adj.P.Val < 0.05), logFC_correlation = 1),
  compare(fit_plate, "plate in the model"),
  compare(fit_keep,  "outliers removed")
)

**🔍 Read the output:** if the counts are similar and the correlations close to **1**, the findings are **robust** to these choices: batch effects and outliers aren't driving them. If adding plate changes a lot, batch effects matter, and plate should be in the main model.

In our test run the answer was more interesting:

| Analysis | Significant proteins | logFC correlation with main |
|---|---|---|
| main | 283 | 1 |
| plate in the model | 182 | 0.93 |
| outliers removed | 533 | 0.995 |

- **Plate in the model:** fewer significant proteins, but the effects themselves changed little (correlation 0.93). Part of the drop is **power**: plate adds about 119 extra terms to a model of only 600 people. If cases and controls were spread unevenly over plates, plate could also be a **confounder**. Check with `count(samples, PlateID, group)`.
- **Outliers removed:** the effects barely changed (0.995), but almost **twice** as many proteins became significant. A few extreme samples were adding a lot of noise.

**💬 Discuss:** which result would you report? (Not the one with the most findings. Pick a rule before looking, report it as the main analysis, and show the others as sensitivity analyses. Here, the **direction and size** of the effects are robust; the **number** called significant isn't.)

👩‍🏫 **Instructor tip:** sensitivity analyses are a good habit to teach early: they're how you show a reviewer that a result doesn't depend on one arbitrary choice.

### Optional: which biological processes?

The same **gene set enrichment** as in 4.5 (`goana()` and Gene Ontology), using each protein's gene. One caution: the **universe** here is the ~5,400 proteins on the Olink panel, which was **designed** around proteins of medical interest, not a random sample of all genes. Enrichment results are relative to that panel.

In [ ]:
for (pkg in c("org.Hs.eg.db", "GO.db")) {
  if (!requireNamespace(pkg, quietly = TRUE)) BiocManager::install(pkg, lib = user_lib, update = FALSE, ask = FALSE)
}

# Each protein's gene, as an Entrez ID (via its UniProt ID)
uniprot <- protein_info$UniProt[match(rownames(fit), protein_info$OlinkID)]
entrez  <- AnnotationDbi::mapIds(org.Hs.eg.db::org.Hs.eg.db, keys = str_remove(uniprot, "_.*"),
                                 keytype = "UNIPROT", column = "ENTREZID", multiVals = "first")
mean(!is.na(entrez))   # share of proteins matched to a gene

go <- goana(fit, coef = "groupcase", geneid = entrez, FDR = 0.05)
topGO(go, ontology = "BP", number = 15)

**🔍 Read the output:** the top biological processes among proteins **up** or **down** in diabetes (columns as in 4.5). Do they match the organ story in the "What we found" table: kidney, fat tissue and lipids, gut, immune response?

In our test run, the top terms were again **up** in diabetes, in two themes: **inflammation and defense** (*defense response*, *inflammatory response*) and **metabolism** (*carboxylic acid*, *organic acid*, *small molecule metabolic process*: the kind of work the liver, kidney, and fat tissue do). With hundreds of significant proteins, these p-values (about 10⁻⁸ to 10⁻⁹) are stronger than for RNA, but broad terms like *response to stimulus* say little on their own.

Two messages you may see: *'select()' returned 1:many mapping between keys and columns* is harmless (a few IDs match more than one gene; `multiVals = "first"` keeps the first). *No DE genes* means nothing passed the FDR threshold, so there's nothing to test for enrichment.

### ✍️ Your turn

1. **Leptin** (`LEP`) is made by **fat tissue**: more body fat, more leptin. Is it higher in people with diabetes? What might explain that? **Challenge:** add each person's **BMI** to the model and look again.
2. Look up ***TCF7L2***: is it on the Olink panel at all?

<details>
<summary><b>Show answer</b></summary>

```r
results %>% filter(Assay %in% c("LEP", "TCF7L2")) %>% select(Assay, logFC, adj.P.Val)
```

1. Leptin is usually **higher** in people with type 2 diabetes, largely because they have more body fat on average. That's **confounding by body size**: to separate diabetes from obesity, you'd add **BMI** (from the physical measurements in the Controlled Tier dataset) to the model.
**Challenge:** BMI is concept `3038553` in the `measurement` table. We average each person's values, dropping impossible ones. 💰 About $1 (it scans the large `measurement` table).

```r
bmi_sql <- str_glue("
SELECT person_id, AVG(value_as_number) AS bmi
FROM `{CURRENT_DATASET}.measurement`
WHERE measurement_concept_id = 3038553
  AND value_as_number BETWEEN 10 AND 100
  AND person_id IN ({str_c(samples$person_id, collapse = ', ')})
GROUP BY person_id")

bmi <- run_query(bmi_sql) %>% mutate(person_id = as.character(person_id))
with_bmi  <- samples %>% left_join(bmi, by = "person_id")
has_value <- !is.na(with_bmi$bmi)

design_bmi <- model.matrix(~ group + age + sex_at_birth + bmi, data = with_bmi[has_value, ])
fit_bmi    <- eBayes(lmFit(npx[, has_value], design_bmi))

topTable(fit_bmi, coef = "groupcase", number = Inf) %>%
  rownames_to_column("OlinkID") %>%
  left_join(protein_info, by = "OlinkID") %>%
  filter(Assay == "LEP") %>%
  select(Assay, logFC, adj.P.Val)
```

If leptin's logFC shrinks toward 0 after adding BMI, its link to diabetes was mostly **body size**.

2. If no row comes back, *TCF7L2* isn't measured. It's a **transcription factor**: it works inside the cell's nucleus and isn't released into the blood, so a blood protein panel isn't designed to measure it. Another reminder that each omics layer shows only part of the picture.

</details>

In [ ]:
# Your code here

## Part 7. pQTLs

*All of Us* has tested each protein against nearby variants (**cis-pQTLs**), in the same format as the eQTL results in 4.5: one row per protein, with its **strongest** nearby variant. These are summary statistics: no individual data.

Each protein is named in `phenotype_id` as **UniProt ID + `_` + Ensembl gene ID**, e.g. `P09769_ENSG00000000938.13`. To match our results (which have a `UniProt` column), we keep the part **before** the underscore with `str_remove(phenotype_id, "_.*")`.

In [ ]:
pqtl <- read_tsv(file.path(PROT_DIR, "pqtl", "cis_pqtl", "AoU_v9_COMB.cis_pqtl.txt.gz"), show_col_types = FALSE) %>%
  mutate(UniProt = str_remove(phenotype_id, "_.*"))

nrow(pqtl)                            # proteins tested
head(pqtl$phenotype_id, 3)            # UniProt_EnsemblID
mean(pqtl$UniProt %in% protein_info$UniProt)   # share that match our proteins: should be close to 1

**🔍 Read the output:** about 5,000 proteins were tested, and almost all of them match a protein in our data.

### Do our top proteins have pQTLs?

In [ ]:
results %>%
  filter(adj.P.Val < 0.05) %>%
  head(20) %>%
  inner_join(pqtl, by = "UniProt") %>%
  select(Assay, logFC, adj.P.Val, variant_id, af, slope, qval) %>%
  arrange(qval)

**🔍 Read the output:** for our top diabetes-related proteins, each one's **strongest** nearby variant. A small `qval` means the protein has a clear **pQTL**: its level is partly set by **genetics**.

In our test run, 18 of the top 20 proteins were in the pQTL file, and almost all had very strong pQTLs (some `qval`s print as 0: too small to store). Proteins are often **much more** strongly controlled by nearby variants than diseases are: compare these p-values with rs7903146's in 4.2. *UMOD* is a good example to discuss: its pQTL is in the *UMOD* gene on chromosome 16, a region also linked to **kidney function** in large GWAS.

**💬 Discuss:** if a protein is higher in diabetes **and** has a strong pQTL, could you use the pQTL to ask whether the protein **causes** diabetes? (This is the idea behind **Mendelian randomization**: genetic variants are fixed at birth, so if variants that raise a protein also raise diabetes risk in a GWAS, the protein may be on the causal path. It needs careful checks, but it's how pQTLs are often used.)

### Does rs7903146 affect any protein?

The **fine-mapped** pQTL file lists, for each protein, the variants most likely to be the **causal** pQTL (**credible sets**), with gene names. We look for rs7903146 by position.

In [ ]:
finemapped <- data.table::fread(
  file.path(PROT_DIR, "pqtl", "finemapped_pqtl", "AoU_COMB_pQTL_SusieMerged.annotated.tsv"),
  select = c("chromosome", "position", "variant", "molecular_trait_id", "gene_name", "pip", "posterior_mean")
) %>% as_tibble()

nrow(finemapped)

finemapped %>%
  filter(position == 112998590, str_detect(as.character(chromosome), "10"))

**🔍 Read the output:** each row is a protein for which rs7903146 is in a **credible set**. `pip` (*posterior inclusion probability*) is the chance that this variant is the causal one, from 0 to 1; `posterior_mean` is its estimated effect.

If the table is **empty** (as in our test run), rs7903146 isn't a likely cause of any measured protein's level. That fits 4.5: rs7903146 has little effect on blood RNA, and its main effects are thought to be in the **pancreas**, on insulin secretion.

## ⚠️ Common errors

| Message | Means |
|---|---|
| *stopifnot(!anyNA(col_nums)) is not TRUE* | A column name in the protein file has changed. Look at `npx_columns` and edit `wanted` |
| `fread` returns 0 rows | The IDs in `ids.txt` didn't match `ResearchID`. Check they're text, not numbers like `1e+06` |
| The awk step takes much longer than 5 minutes | The app may be small or busy. It has to read the whole 2.6 GB file once |
| *there is no package called 'limma'* | Rerun the setup cell and wait for the install to finish |
| `inner_join` returns no rows in Part 7 | None of the top proteins matched. Look at `head(pqtl$phenotype_id)`: has the naming changed? |

## ✅ Key takeaways

- **Olink** measures thousands of proteins with antibody pairs and DNA tags; **NPX** is a relative, log₂-scale level
- Big **long-format** files can be filtered on the command line (**awk**) before R reads them
- Use **QC flags**; reshape long → wide for matrix-based tools like limma
- Blood proteins come from the **whole body**, so they often show more disease differences than blood RNA
- **pQTLs** link genetics to protein levels, and can help ask whether a protein is **causal**

**Next:** **4.7 rnaseq_proteomics**: for people with **both** RNA and protein data, do a gene's RNA and its protein go up and down together?

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).